## Step 1 — Install dependencies and restart runtime

Run this cell, wait for it to finish, then go to **`Runtime → Restart session`**.  
After the restart, **skip this cell** and continue from Step 2.

## Step 1 — Install dependencies and restart runtime

Run this cell first. It pins numpy to avoid a binary incompatibility with scikit-learn, then **automatically restarts the runtime**. After the restart, **skip this cell** and continue from Step 2.

In [ ]:
%%capture
!pip install \
    'numpy==1.24.4' \
    'scikit-learn==1.3.2' \
    'imbalanced-learn==0.11.0' \
    polars pyarrow tqdm \
    pytorch-lightning torchmetrics \
    xgboost shap

In [ ]:
!pip install \
    'scikit-learn==1.4.2' \
    'imbalanced-learn==0.12.4' \
    'pytorch-lightning==1.9.5' \
    'torchmetrics==0.11.4' \
    polars pyarrow tqdm \
    xgboost shap

print('\nAll done — go to Runtime → Restart session, then continue from Step 2.')

In [ ]:
!git clone https://github.com/Vict5/smart-meter-fraud.git

import os, sys
REPO_PATH = '/content/smart-meter-fraud'
os.chdir(REPO_PATH)
sys.path.insert(0, REPO_PATH)

print('Working directory:', os.getcwd())

## Step 3 — Upload raw data files

Upload all 6 raw CSV files: `CONSUMI.csv`, `ANAGRAFICA.csv`, `LAVORI.csv`, `INTERRUZIONI.csv`, `PAROLE_DI_STATO.csv`, `LABELS.csv`.

In [ ]:
import os
from google.colab import files

os.makedirs('data/raw', exist_ok=True)

print('Select all 6 raw CSV files in the upload dialog...')
uploaded = files.upload()

for filename, data in uploaded.items():
    dest = os.path.join('data/raw', filename)
    with open(dest, 'wb') as f:
        f.write(data)
    print(f'  saved -> {dest}')

print('\nFiles in data/raw/:', os.listdir('data/raw'))

## Step 4 — Preprocess raw data

Reads from `data/raw/` and writes cleaned CSV files to `data/processed/`.

In [ ]:
import os, pathlib

os.makedirs('data/processed', exist_ok=True)
os.makedirs('src/models/scalers', exist_ok=True)

from src.features.build_features import PreprocessDataset
preprocessor = PreprocessDataset(pathlib.Path('data/raw'))
preprocessor.preprocess_all()

print('Preprocessing done. Files in data/processed/:', os.listdir('data/processed'))

## Step 5 — (Optional) Restore cached models from Drive

Skip on the first run. On later sessions, uncomment and run to restore
previously trained models and skip ~1 hour of autoencoder training.

In [ ]:
# from google.colab import drive
# import shutil
# drive.mount('/content/drive')
# CACHE = '/content/drive/MyDrive/smart-meter-fraud-cache'
# shutil.copytree(f'{CACHE}/models',              'models',                   dirs_exist_ok=True)
# shutil.copytree(f'{CACHE}/embeddings',          'data/embeddings',          dirs_exist_ok=True)
# shutil.copytree(f'{CACHE}/combined_embeddings', 'data/combined_embeddings', dirs_exist_ok=True)
# print('Models restored from Drive.')

## Step 6 — Enable inline plots

In [ ]:
%matplotlib inline
import matplotlib
matplotlib.rcParams['figure.dpi'] = 100

## Step 7 — Run the pipeline

First run trains all 5 autoencoders (~1 hr on GPU). Later runs detect saved files and skip straight to classification.

| Mode | Description |
|------|-------------|
| `XGBoost` | 4-fold CV XGBoost ensemble (recommended) |
| `RandomForest` | 4-fold CV Random Forest ensemble |
| `DNN_Classifier` | 4-fold CV MLP ensemble |
| `Binary_Classifier` | Threshold anomaly detection (requires `ONLY_REGULAR=True` in config) |

In [ ]:
from src.pipeline import main
main(mode='XGBoost')

In [ ]:
# Other modes:
# main(mode='RandomForest')
# main(mode='DNN_Classifier')
# main(mode='Binary_Classifier')  # only when ONLY_REGULAR=True in config

## Step 8 — (Optional) Save trained models to Drive

Run after the first successful pipeline run to cache models for future sessions.

In [ ]:
# from google.colab import drive
# import shutil
# drive.mount('/content/drive')
# CACHE = '/content/drive/MyDrive/smart-meter-fraud-cache'
# shutil.copytree('models',                   f'{CACHE}/models',              dirs_exist_ok=True)
# shutil.copytree('data/embeddings',          f'{CACHE}/embeddings',          dirs_exist_ok=True)
# shutil.copytree('data/combined_embeddings', f'{CACHE}/combined_embeddings',  dirs_exist_ok=True)
# print('Models saved to Drive.')

## (Optional) Override config hyperparameters

In [ ]:
# Quick smoke-test with fewer epochs — run this BEFORE Step 7
# from src.models.config import Config
# Config.N_EPOCHS_AE = 50
# Config.N_EPOCHS_DNN = 100